In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset, ClassLabel
import transformers

/Users/arslanniazgulov/Documents/nlp_projects/ticket_routing_service/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RANDOM_SEED = 42

os.makedirs(PROJECT_ROOT / "data", exist_ok=True)

In [3]:
dataset = load_dataset(
   "AutoIntent/banking77",
    cache_dir= PROJECT_ROOT / "data"
)

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['utterance', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['utterance', 'label'],
        num_rows: 3080
    })
})

In [5]:
import fastparquet
intents = fastparquet.ParquetFile(PROJECT_ROOT/"data"/"intents.parquet").to_pandas().set_index('id')

In [6]:
intents = intents[['name', 'description']]
NUM_CLASSES = intents.shape[0]

id2label = intents.to_dict()['name']
label2id = {label: ind for ind, label in id2label.items()}


In [ ]:
import json

with open('id2label.json', "w") as f:
    json.dump(id2label, fp=f)

In [ ]:
train_dataset = dataset['train']
test_dataset = dataset['test']

In [ ]:
train_dataset_casted = train_dataset.cast_column("label", ClassLabel(num_classes=intents.shape[0], names=intents['name'].to_list()))
train_val_dataset = train_dataset_casted.train_test_split(test_size=0.2, shuffle=True, stratify_by_column='label', seed=RANDOM_SEED)

In [ ]:
train_dataset_splited = train_val_dataset['train']
val_dataset_splited = train_val_dataset['test']

### Анализ Данных

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

In [ ]:
counter = Counter(train_dataset_splited['label'])

print("Самый популярный класс: ", counter.most_common(5))
print("Самый редкий класс: ", counter.most_common()[:-5:-1])


In [ ]:
UTTERANCE = 'utterance'
LABEL = 'label'

In [ ]:
def add_len(ex):
    ex["length"] = len(ex[UTTERANCE])
    return ex

train_dataset_splited = train_dataset_splited.map(add_len)

In [ ]:
length_counter = Counter(train_dataset_splited['length'])

In [ ]:
print("Самые частые length (длина, кол-во):", *length_counter.most_common(5))
print("Самые редкие length (длина, кол-во):", *length_counter.most_common()[:-10:-1])

In [ ]:
label_length_df = pd.DataFrame({
    "label": train_dataset_splited['label'],
    "length": train_dataset_splited['length']
})

In [ ]:
label_length_df.groupby("label").aggregate(["min", "max", "mean", "count"])

# BaseLine

In [ ]:
X_train = train_dataset_splited[UTTERANCE]
y_train = train_dataset_splited[LABEL]

X_val = val_dataset_splited[UTTERANCE]
y_val = val_dataset_splited[LABEL]

X_test = test_dataset[UTTERANCE]
y_test = test_dataset[LABEL]

In [ ]:
len(Counter(y_train)), len(Counter(y_val))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score, accuracy_score

In [ ]:
baseline = Pipeline(
    [("tfidf", TfidfVectorizer(
        min_df=1,
        max_df=0.9

    )),
    ("log_reg", LogisticRegression(
    )
    )

    ]
)

In [ ]:
len(y_val)

In [ ]:
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)
print("macro F1:   ", f1_score(y_test, y_pred, average="macro"))
print("micro F1:   ", f1_score(y_test, y_pred, average="micro"))
print("weighted F1:", f1_score(y_test, y_pred, average="weighted"))
print("accuracy:   ", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
import mlflow
import mlflow.sklearn

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")
mlflow.set_experiment("banking77-intent")

In [ ]:
with mlflow.start_run(run_name="baseline-tfidf-logreg") as run:
    # теги
    mlflow.set_tags({
        "stage": "baseline",
        "task": "intent-classification",
        "dataset": "banking77",
    })

    # параметры
    mlflow.log_params({
        "vectorizer": "tfidf(1,2)_min_df2_sublinear",
        "clf": "LogisticRegression",
        "C": 1.0,
        "class_weight": "balanced",
        "max_iter": 2000,
        "test_size": 0.2,
        "seed": 42,
        "n_train": len(X_train),
        "n_val": len(X_val),
    })

    # обучение
    baseline.fit(X_train, y_train)
    y_pred = baseline.predict(X_val)

    # метрики
    mlflow.log_metrics({
        "macro_f1":    f1_score(y_val, y_pred, average="macro"),
        "micro_f1":    f1_score(y_val, y_pred, average="micro"),
        "weighted_f1": f1_score(y_val, y_pred, average="weighted"),
        "accuracy":    accuracy_score(y_val, y_pred),
    })


    # модель
    mlflow.sklearn.log_model(baseline, name="model", serialization_format="pickle")

    print("run_id:", run.info.run_id)

In [ ]:
run_id = "625da7ed2b9b4537a6fb9fafe40120de"
loaded = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
loaded.predict(["getting virtual card", "", "как дела"])

In [ ]:
id2label[40]

### Трансформер based

In [64]:
from transformers import (AutoTokenizer,
                        AutoModelForSequenceClassification,
                        TrainingArguments,
                        Trainer,
                        DataCollatorWithPadding,
                        DistilBertTokenizer,
                        DistilBertForSequenceClassification
                        )

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 100

tokenizer = DistilBertTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=PROJECT_ROOT / "transformers" / MODEL_NAME / "tokenizer"
    )

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    cache_dir=PROJECT_ROOT / "transformers" /  MODEL_NAME / "model",
    num_labels = NUM_CLASSES,
    id2label = id2label,
    label2id = label2id,
    attn_implementation="eager"  # <-- Добавьте эту строку

)

In [ ]:
print(model.config.num_labels)
print(model.config.id2label[0])
print(model.classifier.out_features)

In [ ]:
def tokenize(batch):
    return tokenizer(batch[UTTERANCE], truncation=True, max_length=MAX_LEN)

In [ ]:
val_dataset_splited

In [ ]:
tok_train = train_dataset_splited.map(tokenize, batched=True).remove_columns([UTTERANCE, 'length'])
tok_val = val_dataset_splited.map(tokenize, batched=True).remove_columns([UTTERANCE])
tok_test = train_dataset_splited.map(tokenize, batched=True).remove_columns([UTTERANCE, 'length'])

In [ ]:
tok_val['label']

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1":        f1_score(labels, preds, average="macro",        zero_division=0),
        "weighted_f1":     f1_score(labels, preds, average="weighted",     zero_division=0),
        "recall_macro":    recall_score(labels, preds, average="macro",    zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "accuracy":        accuracy_score(labels, preds),
    }

In [ ]:
import torch
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer

cw = compute_class_weight(
    "balanced",
    classes=np.unique(tok_train['label']),
    y=tok_train['label']
)
class_weights = torch.tensor(cw, dtype=torch.float)
class_weights

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained("/Users/arslanniazgulov/Documents/nlp_projects/ticket_routing_service/ft_models/distilbert-base-uncased")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"runs/hf/{MODEL_NAME}",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",         
    seed=RANDOM_SEED,
)

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

### LoRa

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = NUM_CLASSES,
    cache_dir= PROJECT_ROOT / 'transformers' / MODEL_NAME / 'model'
)

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=[
        "q_lin",           # Query в attention
        "k_lin",           # Key в attention
        "v_lin",           # Value в attention
        "out_lin",         # Output projection
        "lin1",            # FFN intermediate (вместо w1)
        "lin2",            # FFN output (вместо w2)
    ],
)

model_lora = get_peft_model(model, lora_config)
model_lora.print_trainable_parameters()

In [ ]:
model

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"runs/hf/{MODEL_NAME}",
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="mlflow",         
    seed=RANDOM_SEED,
)

trainer = WeightedTrainer(
    model=model_lora,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
model_lora.save_pretrained(PROJECT_ROOT / "fit_models" / (MODEL_NAME + "_lora_10epoch"))